# Listing photos and views: Williams test after ANOVA

**One-Way Analysis of Variance** · a worked example from the StatNex teaching guide

**The story.** A resale app looked at 21 listings for the same kind of jacket, posted with 1, 3, 5 or 10 photos, and counted how many people viewed each listing in its first week.

**The question.** Do listings with more photos get more views, and how many photos are enough?

**To run it**, choose **Runtime → Run all**. Colab first warns that this notebook was not written by Google; it reads none of your data, so choose **Run anyway**. The first cells save the table in this session. The rest is the calculation StatNex runs, written out in Python. It does not use StatNex, and every number it prints was checked against the app's.

## The table

This cell saves the table `06_nonparametric/kruskal_wallis.csv` (6 rows) as `data.csv`, the file the code reads.

In [ ]:
%%writefile data.csv
1 photo,3 photos,5 photos,10 photos
88.0,118.0,135.0,148.0
102.0,131.0,150.0,170.0
95.0,109.0,128.0,139.0
120.0,140.0,162.0,165.0
76.0,,141.0,158.0
110.0,,155.0,

These cells save `welch_anova.py`, `williams.py`: steps that no library does, written out. The analysis imports them.

In [ ]:
%%writefile welch_anova.py
"""Brown and Forsythe's (1974) one-way ANOVA of means, which does not assume one variance.

Welch's test comes from statsmodels' anova_oneway(use_var="unequal"). Its use_var="bf"
applies Mehrotra's (1997) correction to the first degrees of freedom, and SciPy has no
Brown-Forsythe test of means, so the original test is written out here.
"""

import numpy as np
from scipy import stats


def brown_forsythe(groups):
    """Brown & Forsythe (1974) F*: (F, df1, df2, P), denominator weighted by variance."""
    k = len(groups)
    n = np.array([len(g) for g in groups], dtype=float)
    means = np.array([np.mean(g) for g in groups])
    variances = np.array([np.var(g, ddof=1) for g in groups])
    grand = np.concatenate(groups).mean()
    # The usual between-groups sum of squares, set against each group's variance
    # weighted by 1 - n_i / N instead of one pooled variance.
    numerator = (n * (means - grand) ** 2).sum()
    weights = (1 - n / n.sum()) * variances
    f = numerator / weights.sum()
    # Satterthwaite's degrees of freedom for that weighted denominator.
    c = weights / weights.sum()
    df1, df2 = k - 1, 1 / ((c**2 / (n - 1)).sum())
    return f, float(df1), df2, stats.f.sf(f, df1, df2)

In [ ]:
%%writefile williams.py
"""Williams' (1971, 1972) test for a dose response that only goes one way.

No Python library has it. Two pieces are written out: the isotonic estimate of each
dose's mean, and the critical value. Williams' printed table assumes equal dose groups;
this computes the value for the group sizes the design has -- the same numerical recipe
StatNex uses, so the constants agree with the app's to the last digit.

How: under the null hypothesis the differences D_u = (pooled mean of doses u..j) -
(control mean) have the covariance of one Brownian motion read at the times
1/n_0 + 1/N_u, N_u being the doses u..j pooled. The statistic stays below c exactly
when that motion stays below a line, a chance carried back from the last time to the
first on a grid, one convolution per dose. R's mvtnorm computes the same probability
with Miwa's and Genz's algorithms (pmvnorm, pmvt) and agrees to within 4e-6.
"""

import functools

import numpy as np
from scipy import signal, special, stats

# The grid: distance below the line, in standard errors, up to REACH. Its step is GRID, or an
# eighth of the walk's smallest step where that is finer (down to FINEST): a small step
# rises from 0 within a few sd of the line, and a coarser grid draws that as a straight line.
GRID, FINEST, REACH = 1e-3, 1e-4, 10.0


def isotonic_extreme(sizes, means, step, increasing):
    """The largest (or smallest) pooled mean of any block of doses ending at ``step``.

    ``sizes`` and ``means`` are ordered control first, then the doses.
    """
    pooled = [
        float((sizes[u:step + 1] * means[u:step + 1]).sum() / sizes[u:step + 1].sum())
        for u in range(1, step + 1)
    ]
    return max(pooled) if increasing else min(pooled)


@functools.lru_cache(maxsize=8)
def _scale_nodes(nu):
    """Gauss-Legendre nodes for E[...] over U = S/sigma, where nu*U^2 ~ chi-squared(nu)."""
    lo = np.sqrt(stats.chi2.ppf(1e-13, nu) / nu)
    hi = np.sqrt(stats.chi2.isf(1e-13, nu) / nu)
    x, w = special.roots_legendre(120)
    u = 0.5 * (hi - lo) * x + 0.5 * (hi + lo)
    log_f = ((nu / 2 - 1) * np.log(nu * u**2) - nu * u**2 / 2 - (nu / 2) * np.log(2.0)
             - special.gammaln(nu / 2) + np.log(2 * nu * u))
    wu = w * 0.5 * (hi - lo) * np.exp(log_f)
    return u, wu / wu.sum()


def _ramp(x, sd):
    """E[max(x - D, 0)] for D ~ N(0, sd^2): the normal CDF integrated once."""
    return x * stats.norm.cdf(x / sd) + sd * stats.norm.pdf(x / sd)


def _below_line(q, sd, d):
    """E[Q(h - D)] for D ~ N(0, sd^2), at h = 0, d, 2d, ...

    Q is straight lines between its nodes, 0 below h = 0 and 1 past the last node; each
    node's tent is integrated against the normal density exactly.
    """
    half = int(np.ceil(9.0 * sd / d)) + 2
    offsets = np.arange(-half, half + 1) * d
    tent = (_ramp(offsets + d, sd) - 2.0 * _ramp(offsets, sd) + _ramp(offsets - d, sd)) / d
    padded = np.concatenate([np.zeros(half), q, np.ones(half)])
    smoothed = signal.fftconvolve(padded, tent, mode="same")[half:half + q.size]
    # The node at h = 0 has no left half: Q is 0 just below it, not a ramp up to q[0].
    h = np.arange(q.size) * d
    left_half = (_ramp(h + d, sd) - _ramp(h, sd)) / d - stats.norm.cdf(h / sd)
    return smoothed - q[0] * left_half


@functools.lru_cache(maxsize=64)
def _curve(sizes):
    """P(statistic <= b) at infinite df, for b = 0, d, 2d, ..., and the grid step d.

    ``sizes`` is (control, dose 1, ..., the dose compared).
    """
    control = sizes[0]
    doses = np.asarray(sizes[1:], float)
    pooled = np.cumsum(doses[::-1])[::-1]
    times = (1.0 / control + 1.0 / pooled) / (1.0 / control + 1.0 / doses[-1])
    sds = np.sqrt(np.diff(times, prepend=0.0))   # sqrt(t_1), sqrt(t_2 - t_1), ...
    d = max(FINEST, min(GRID, float(sds.min()) / 8.0))
    q = np.ones(int(round(REACH / d)) + 1)
    for u in range(doses.size - 1, 0, -1):
        q = _below_line(q, float(sds[u]), d)
    return _below_line(q, float(sds[0]), d), d


def critical_value(sizes, step, nu, alpha):
    """Williams' constant for comparing dose ``step`` with the control.

    ``sizes`` are the group sizes, control first; ``nu`` the error degrees of freedom
    (``np.inf`` for the rank form, Shirley-Williams).
    """
    curve, d = _curve(tuple(float(s) for s in sizes[:step + 1]))
    grid = np.arange(curve.size) * d
    u, wu = (np.array([1.0]), np.array([1.0])) if np.isinf(nu) else _scale_nodes(float(nu))

    def tail(c):
        return 1.0 - float(wu @ np.interp(c * u, grid, curve, right=1.0))

    lo, hi = 0.0, 8.0
    while tail(hi) > alpha:   # very small nu: the one-dose 2.5% point at nu = 1 is 12.7
        lo, hi = hi, 2.0 * hi
    for _ in range(60):
        mid = 0.5 * (lo + hi)
        lo, hi = (mid, hi) if tail(mid) > alpha else (lo, mid)
    return 0.5 * (lo + hi)

## Libraries

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from scipy import stats
from statsmodels.stats.anova import anova_lm
from statsmodels.stats.oneway import anova_oneway
from welch_anova import brown_forsythe
from williams import critical_value, isotonic_extreme

pd.set_option("display.width", 120)

## The data

In [ ]:
# One column per group. Blank cells are skipped, so groups may differ in size.
data = pd.read_csv("data.csv", keep_default_na=False, na_values=[""])
levels = ["1 photo", "3 photos", "5 photos", "10 photos"]
groups = {name: data[name].dropna().to_numpy(float) for name in levels}
samples = list(groups.values())

## Descriptive statistics

In [ ]:
n = np.array([len(v) for v in samples], dtype=float)
means = np.array([v.mean() for v in samples])
sds = np.array([v.std(ddof=1) for v in samples])
ses = sds / np.sqrt(n)
descriptive = pd.DataFrame(
    {"n": n.astype(int), "mean": means, "SD": sds, "mean - SD": means - sds,
     "mean + SD": means + sds, "SE": ses, "mean - SE": means - ses, "mean + SE": means + ses},
    index=levels,
)
print(descriptive)

## Test for homogeneity of variance

In [ ]:
# The ANOVA assumes every group has the same spread. Both tests ask whether the spreads
# could be equal: a small P value says they probably are not. Bartlett's test assumes
# normal data; Levene's test copes better with non-normal data. center="mean" gives
# Levene's original form; SciPy's default, the median, is the Brown-Forsythe variant.
bartlett = stats.bartlett(*samples)
levene = stats.levene(*samples, center="mean")
print(f"\nBartlett: chi-square = {bartlett.statistic:.4f}, df = {len(samples) - 1}, P = {bartlett.pvalue:.4g}")
print(f"Levene:   F = {levene.statistic:.4f}, df = ({len(samples) - 1}, {int(n.sum()) - len(samples)}), "
      f"P = {levene.pvalue:.4g}")

## Analysis of variance table

In [ ]:
# F = (variation between the group means, per df) / (variation within the groups, per
# df). If every population mean were the same, F would be near 1, and P is the chance
# of an F at least this large in that case. The test assumes independent values,
# normal errors and one common spread. statsmodels fits the one-factor linear model.
k, total_n = len(samples), int(n.sum())
long = pd.DataFrame({"value": np.concatenate(samples),
                     "group": np.repeat(levels, [len(v) for v in samples])})
fit = smf.ols("value ~ C(group)", data=long).fit()
table = anova_lm(fit)
ss_between, ss_within = table["sum_sq"].iloc[0], table["sum_sq"].iloc[1]
df_between, df_within = int(table["df"].iloc[0]), int(table["df"].iloc[1])
ms_between, ms_within = table["mean_sq"].iloc[0], table["mean_sq"].iloc[1]
f_value, p_value = table["F"].iloc[0], table["PR(>F)"].iloc[0]
anova = pd.DataFrame(
    [[ss_between, df_between, ms_between, f_value, p_value],
     [ss_within, df_within, ms_within, np.nan, np.nan],
     [ss_between + ss_within, total_n - 1, np.nan, np.nan, np.nan]],
    index=["Between groups", "Error", "Total"], columns=["SS", "df", "MS", "F", "P"],
)
print()
print(anova)

## Tests that do not assume equal variances

In [ ]:
# Read these instead of the table above when the variances differ. statsmodels'
# anova_oneway(use_var="unequal") is Welch's test. Its use_var="bf" applies Mehrotra's
# (1997) correction to the first df, so Brown and Forsythe's original test is written
# out in welch_anova.py.
welch = anova_oneway(long["value"], long["group"], use_var="unequal")
unequal = pd.DataFrame(
    [[welch.statistic, welch.df[0], welch.df[1], welch.pvalue], brown_forsythe(samples)],
    index=["Welch", "Brown-Forsythe"], columns=["F", "df1", "df2", "P"],
)
print()
print(unequal)

## Multiple comparison: Williams

In [ ]:
# Each dose against the control, assuming the response moves one way as the dose
# rises. A dose's mean is replaced by the isotonic estimate (neighbouring means pooled
# until they rise in order), and the statistic is compared with Williams' critical
# value for these group sizes; * at 5%, ** at 2.5% (one-sided). The test steps down from
# the top dose and stops at the first dose that is not significant, at each level, so no
# lower dose is marked after that. Neither SciPy nor statsmodels has Williams' test, so
# it is written out in williams.py.
# The control is the level the settings name, spaces at either end aside, as the app
# reads it; the first level when they name none.
named = ("1 photo" or "").strip()
control = next((name for name in levels if name.strip() == named), levels[0])
others = [name for name in levels if name != control]
order = [control, *others]
dose_n = np.array([len(groups[name]) for name in order], dtype=float)
dose_means = np.array([groups[name].mean() for name in order])
rows = []
for label, increasing in (("control ≤", True), ("control ≥", False)):
    open5 = open25 = True
    for step in range(len(others), 0, -1):
        name = order[step]
        estimate = isotonic_extreme(dose_n, dose_means, step, increasing)
        difference = estimate - dose_means[0] if increasing else dose_means[0] - estimate
        se = np.sqrt(ms_within * (1 / dose_n[step] + 1 / dose_n[0]))
        statistic = difference / se
        crit5, crit25 = (critical_value(dose_n, step, df_within, a) for a in (0.05, 0.025))
        open25 = open25 and statistic >= crit25
        open5 = open5 and statistic >= crit5
        mark = "**" if open25 else "*" if open5 else ""
        rows.append([label, control, name, dose_means[0], estimate, difference, se, statistic,
                     crit5, crit25, mark])
pairs = pd.DataFrame(rows, columns=["alternative", "level 1", "level 2", "mean 1", "mean 2",
                                    "difference", "SE", "statistic", "critical 5%",
                                    "critical 2.5%", ""])
print()
print(pairs.to_string(index=False))

## What the result says

One-way analysis of variance (ANOVA) compares the mean views of the four photo counts. The means rise from 98.5 with 1 photo to 124.5, 145.2 and 156 with 3, 5 and 10 photos. F, the ratio of the variation between groups to the variation inside groups, is 18.86, and its P value is 0.00001. The P value is the chance of gaps this large if photos made no difference. The test for homogeneity of variance (P = 0.965) gives no sign that the groups differ in spread.

The Williams test then compares each photo count with the control, 1 photo, assuming that mean views do not fall as photos rise. The statistics are 6.86 for 10 photos, 5.84 for 5 photos and 2.91 for 3 photos. All are above their 2.5% critical values, which lie between 2.1 and 2.2.

Here even three photos beat one, by 26 views on average. The test does not compare ten photos with five.

## Your own data

Open the **Files** panel (the folder icon on the left) and upload a CSV file. Then change `"data.csv"` in the `read_csv` line to its name, change the column names in the code to yours, and choose **Run all** again.